In [7]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import heapq

**7.5 An Inventory Model**

Consider a shop that stocks a particular type of product that it sells for a price of r per unit. Customers demanding this product appear in accordance with a Poisson process with rate λ, and the amount demanded by each one is a random variable having distribution G. In order to meet demands, the shopkeeper must keep an amount of the product on hand, and whenever the on-hand inventory becomes low, additional units are ordered from the distributor. The shopkeeper uses a so-called (s, S) ordering policy; namely, whenever the on-hand inventory is less than s and there is no presently outstanding order, then an amount is ordered to bring it up to S, where s < S. That is, if the present inventory level is x and no order is outstanding, then if x < s the amount S − x is ordered. The cost of ordering y units of the product is a speciﬁed function c(y), and it takes L units of time until the order is delivered, with the payment being made upon delivery. In addition, the shop pays an inventory holding cost of h per unit item per unit time. Suppose further that whenever a customer demands more of the product than is presently available, then the amount on hand is sold and the remainder of the order is lost to the shop.

Let us see how we can use simulation to estimate the shop’s expected proﬁt up to some ﬁxed time T . To do so, we start by deﬁning the variables and events as follows.


In [24]:
class InventoryStockSimulator:
    def __init__(self, custumer_function, demand_function, order_cost_function, L, r, s, S, h, x, closing_time):
        
        # Parameters
        self.custumer_function = custumer_function
        self.demand_function = demand_function
        self.order_time_arrival = L
        self.order_cost_function = order_cost_function
        self.sell_price_per_unit = r
        self.order_policy = { 's': s, 'S': S}
        self.holding_cost_per_unit = h
        self.closing_time = closing_time
        self.initial_inventory_amount = x
        
       

    
    def initialize_variables(self):
        # System State Variables
        self.inventory_amount = self.initial_inventory_amount
        self.order_amount = 0
        
        # Counter Variables
        self.total_order_amount = 0
        self.total_inventory_holding_cost = 0
        self.total_revenue = 0
        self.total_lost_order_amount = 0
        
        
        # Events Variables
        self.time = 0
        self.events_queue = []
    
    def simulate(self):
        
        self.initialize_variables()
        
        heapq.heappush(self.events_queue, (self.custumer_function(), 'custumer'))
        
        while self.time < self.closing_time:
            current_time, event_type = heapq.heappop(self.events_queue)
            
            if event_type == "custumer":
                
                self.total_inventory_holding_cost = self.total_inventory_holding_cost + (current_time - self.time) * self.inventory_amount * self.holding_cost_per_unit
                self.time = current_time
                
                demand = self.demand_function()
                w = min(demand, self.inventory_amount)
                self.total_revenue += w * self.sell_price_per_unit
                self.total_lost_order_amount += (demand - w)*self.sell_price_per_unit
                self.inventory_amount -= w
                
                if self.inventory_amount < self.order_policy['s'] and self.order_amount == 0:
                    self.order_amount = self.order_policy['S'] - self.inventory_amount
                    heapq.heappush(self.events_queue, (current_time + self.order_time_arrival, 'order'))
                    
                heapq.heappush(self.events_queue, (current_time + self.custumer_function(), 'custumer'))
            
            else:
                self.total_inventory_holding_cost += (current_time - self.time) * self.inventory_amount * self.holding_cost_per_unit
                self.time = current_time
                
                self.total_order_amount += self.order_cost_function(self.order_amount)
                self.inventory_amount += self.order_amount
                self.order_amount = 0
                
                if self.inventory_amount < self.order_policy['s'] and self.order_amount == 0:
                    self.order_amount = self.order_policy['S'] - self.inventory_amount
                    heapq.heappush(self.events_queue, (current_time + self.order_time_arrival, 'order'))


In [40]:
def custumer_function(lamb = 0.5): 
    return np.random.exponential(lamb)

def demand_function(low = 1, high = 5):
    return np.random.randint(low, high)
         
def order_cost_function(x):
    return np.exp(np.sqrt(x))

In [41]:
I = InventoryStockSimulator(custumer_function, 
                        demand_function,
                        order_cost_function,
                        L=5,
                        r=1.5,
                        s=10,
                        S=50,
                        h=0.5,
                        x=80,
                        closing_time=100)

I.simulate()
print(f"H : {I.total_inventory_holding_cost}\n",
      f"C : {I.total_order_amount}\n",
      f"R : {I.total_revenue}\n"
      f"P : {I.total_lost_order_amount}\n",
      f"(R - C - H) / T : {(I.average_profit_per_uni)}")

H : 950.4581151071035
 C : 5128.751041608461
 R : 588.0
P : 184.5
 (R - C - H) / T : 0.0
